# Autoresearch

An agent edits one file, measures, keeps or discards — and repeats without you.

*You are the slow part of research, not the thinking.*

> Faithful to [karpathy/autoresearch](https://github.com/karpathy/autoresearch): one loop.

## 0. Setup

Reading and writing are separate permissions — this split *is* the design:

| path | reads | writes |
|---|---|---|
| `program.md` | **yes** — every prompt | no |
| `src/` | **yes** | **yes** — on a win |
| `harness/` | no | no |

In [1]:
import os
import re
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

from harness.measure import measure   # protected/ — never enters a prompt

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI(
    api_key=os.environ["DEEPINFRA_API_KEY"],
    base_url="https://api.deepinfra.com/v1/openai",
)
MODEL = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"   # see Weaknesses: 70B fails this task

SOLVE = Path("src/solve.py")   # the only file the agent may rewrite
PROGRAM = Path("program.md").read_text()
MAX_EXPERIMENTS, PATIENCE = 15, 8

## 1. The boundary — load both and look

The agent reads `program.md` and `src/`. It never sees `harness/`, so the verifier can't be
argued with.

In [2]:
EDITABLE  = sorted(f for f in Path("src").iterdir() if f.is_file())
PROTECTED = sorted(f for f in Path("harness").iterdir() if f.is_file())

for label, paths, seen in [("EDITABLE  src/", EDITABLE, "the agent rewrites these"),
                           ("PROTECTED harness/", PROTECTED, "never enters a prompt")]:
    print(f"{'='*70}\n{label}  — {seen}\n{'='*70}")
    for f in paths:
        print(f"--- {f} ---")
        print(f.read_text().rstrip(), "\n")

EDITABLE  src/  — the agent rewrites these
--- src/solve.py ---
# EDITABLE — autoresearch rewrites this in place. Karpathy's train.py.


def solve(n):
    return sum(i for i in range(n) if i % 3 == 0 or i % 5 == 0) 

PROTECTED harness/  — never enters a prompt
--- harness/cases.json ---
{
  "_note": "n is exclusive: sum over i < n. This is the spec the model keeps getting wrong.",
  "tests": [[10, 23], [100, 2318], [1000, 233168]],
  "workload": 2000000
} 

--- harness/measure.py ---
"""PROTECTED — never enters a prompt. Karpathy's `prepare.py`.

An agent that can edit the verifier optimises the verifier: it deletes the failing
case instead of passing it.
"""

import json
import time
from pathlib import Path

_CASES = json.loads((Path(__file__).parent / "cases.json").read_text())
TESTS = [tuple(t) for t in _CASES["tests"]]   # (n, expected)
WORKLOAD = _CASES["workload"]


def measure(src: str) -> tuple[float | None, str]:
    """(seconds, note). None = rejected. Correct first, then fas

## 2. The prompt

Everything the agent gets: `program.md`, the editable file, the last measurement. Nothing else.

In [3]:
def propose(src, seconds, note):
    """The only thing the agent ever sees: program.md + solve.py + the last measurement."""
    msg = (f"{PROGRAM}\n\n"
           f"src/solve.py:\n```python\n{src}\n```\n"
           f"Current: {seconds*1000:.3f} ms. Last result: {note}\n"
           f"Propose ONE change. Reply with only the new `def solve(n):` in a ```python block.")
    out = client.chat.completions.create(
        model=MODEL, max_tokens=500, messages=[{"role": "user", "content": msg}],
    ).choices[0].message.content
    m = re.search(r"```python\n(.*?)```", out, re.S)
    return m.group(1).strip() if m else None

## 3. The loop

Measure, then write only on a win — so `src/solve.py` only ever holds the best.

In [4]:
def autoresearch():
    best = SOLVE.read_text()
    best_s, note = measure(best)
    stale = 0

    for i in range(1, MAX_EXPERIMENTS + 1):
        cand = propose(best, best_s, note)
        secs, note = measure(cand) if cand else (None, "no code block")
        kept = secs is not None and secs < best_s

        if kept:
            SOLVE.write_text(cand)          # commit: the file only ever holds the best
            best, best_s, stale = cand, secs, 0
        else:
            stale += 1                      # discard: nothing was written, nothing to undo

        print(f"  {i:>2} {('%.3f ms' % (secs*1000)) if secs else '—':>12}  "
              f"{'KEEP ' if kept else 'discard'}  best={best_s*1000:.3f} ms  {note[:44]}")

        if stale >= PATIENCE:
            print(f"  stop: no improvement in {PATIENCE} experiments")
            break
    return best, best_s

## 4. Run

`git diff src/` afterwards to see what it did.

In [5]:
base = SOLVE.read_text()
base_s, _ = measure(base)
print(f"baseline {base_s*1000:.3f} ms\n")

best, best_s = autoresearch()
print(f"\nbaseline {base_s*1000:.3f} ms -> best {best_s*1000:.3f} ms  ({base_s/best_s:.0f}x)")
print(best)

baseline 100.243 ms

   1     0.004 ms  KEEP   best=0.004 ms  ok
   2     0.005 ms  discard  best=0.004 ms  ok
   3     0.004 ms  discard  best=0.004 ms  ok
   4     0.005 ms  discard  best=0.004 ms  ok
   5     0.004 ms  discard  best=0.004 ms  ok
   6     0.003 ms  KEEP   best=0.003 ms  ok
   7     0.005 ms  discard  best=0.003 ms  ok
   8     0.002 ms  KEEP   best=0.002 ms  ok
   9     0.005 ms  discard  best=0.002 ms  ok
  10     0.005 ms  discard  best=0.002 ms  ok
  11     0.007 ms  discard  best=0.002 ms  ok
  12     0.004 ms  discard  best=0.002 ms  ok
  13     0.006 ms  discard  best=0.002 ms  ok
  14     0.007 ms  discard  best=0.002 ms  ok
  15     0.005 ms  discard  best=0.002 ms  ok

baseline 100.243 ms -> best 0.002 ms  (66828x)
def solve(n):
    n -= 1
    sum3 = (n // 3) * (3 + (n // 3) * 3) // 2
    sum5 = (n // 5) * (5 + (n // 5) * 5) // 2
    sum15 = (n // 15) * (15 + (n // 15) * 15) // 2
    return sum3 + sum5 - sum15


## Key findings

The run above, `MAX_EXPERIMENTS=15, PATIENCE=8`, Llama-4-Maverick:

```
baseline 100.243 ms -> best 0.002 ms   (66,828x)
   1   0.004 ms  KEEP     <- the closed form, first try
   6   0.003 ms  KEEP     <- noise
   8   0.002 ms  KEEP     <- noise
  rest  discard, every candidate `ok`
```

**Experiment 1 did all the work.** The other 14 found nothing — and two of them were committed
anyway, because at these speeds the timer measures scheduling, not code.

No candidate was ever `WRONG`, so Maverick never hit the off-by-one that stops
`Llama-3.1-70B` three times running. The task the loop solves is only hard for a weaker model.

## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| **Noise is indistinguishable from progress** | Once fast, the metric is noise: the found solution re-times at `0.0003–0.0018 ms` — a 426% spread. Experiments 6 and 8 "improved" `0.004 → 0.003 → 0.002 ms` and were committed. Both were noise | Min of N runs, and require a real gain (say 5%) to count as a win |
| **Noise also defeats the stopping rule** | `PATIENCE=8` never fired, because those two noise-wins reset the counter. The loop ran all 15 experiments after finding the answer at experiment 1 | Same fix — a win must clear the noise floor |
| **The model is the ceiling** | `Llama-3.1-70B` never solves this: same off-by-one every attempt (`WRONG on n=10: got 33, want 23`). Maverick lands it on experiment 1 | Use a model that can do the task |
| **The boundary is convention** | `exec` runs model code in-process with write access to `harness/` | Subprocess, read-only mount, timeout |
| **The metric is the objective** | It optimises `measure`'s return, not your intent. Correctness is pass/fail — no partial credit | Verify the verifier first |
| **No rollback** | The first win overwrites the baseline, and this dir is untracked | Commit before running |
| **No memory** | Each proposal sees only the best and the last note, so it retries dead ideas | Pass the log |
| **One loop, by design** | The strategy never changes; nothing notices a stall | none — a second loop is a different mechanism |